In [1]:
# Load all metrics npz files from Results/{timestamp}.
# They are named {dataset}_{sampler}_{proposal}_{ss}_metrics.npz beside the
# {dataset}_{sampler}_{proposal}_{ss}.h5 they were evaluated from, where {ss}
# is ss_prop * 100 with any decimal point written as "p" (covtype_smc_DA_12p5).
# MH takes no subsample and is always tagged 100.

import os
import re
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")
plt.rcParams.update({'text.usetex': False})
import matplotlib.patheffects as pe


In [ ]:
import hashlib

def parse_ss(tag):
    """The {ss} field of a filename back to ss_prop: '12p5' -> 0.125, '50' -> 0.5."""
    return float(tag.replace("p", ".")) / 100


def load_config(timestamp, results_root="Results"):
    # load the config json for the given timestamp
    config_path = os.path.join(results_root, timestamp, "config.json")
    with open(config_path, "r") as f:
        config = json.load(f)
    return config


def config_stems(entry):
    """The file stems one config.json entry's runs were written under, built
    the way results_io.experiment_stem names them: an entry may list several
    samplers and proposals, and each pair has its own files."""
    def as_list(value):
        return [value] if isinstance(value, str) else list(value)

    # What the runs started from: nothing for a stump, _init<N> for a random
    # N-node tree, _init-prior for draws from the prior, _init-<name> for a
    # given one -- named by a digest of its rows when the entry does not name
    # it (results_io.init_of). An entry without init_tree_size predates the
    # prior default and started from a stump.
    rows = entry.get("init_tree")
    if rows is None:
        size = entry.get("init_tree_size") or 0
        if isinstance(size, str):
            init = f"_init-{size}"
        else:
            init = f"_init{int(size)}" if size else ""
    else:
        name = entry.get("init_tree_name")
        if not name:
            canonical = sorted([int(r[0]), int(r[1]), int(r[2]), int(r[3]), float(r[4])]
                               for r in rows)
            name = "t" + hashlib.sha1(json.dumps(canonical).encode()).hexdigest()[:8]
        init = f"_init-{name}"

    stems = []
    for sampler in as_list(entry.get("samplers", ())):
        for proposal in as_list(entry.get("proposals", ())):
            # MH takes no subsample and is always tagged 100.
            ss_prop = 1.0 if proposal == "MH" else entry.get("ss_prop")
            if ss_prop is None or "dataset" not in entry:
                continue
            ss = f"{round(100 * ss_prop, 6):g}".replace(".", "p")
            stems.append(f"{entry['dataset']}_{sampler}_{proposal}_{ss}{init}")
    return stems


def load_metrics_files(timestamp, results_root="Results"):
    """(smc, mcmc) lists of dicts, one per *_metrics.npz in the run directory
    that config.json has an experiment for. Each carries that experiment's
    chains, particles (None for MCMC) and max_tree_size; a metrics file no
    experiment in the config accounts for is skipped."""
    results_path = os.path.join(results_root, timestamp)
    PROPOSALS = {"MH": "MH", "DA": "DA", "FlatHINTS": "Flat HINTS", "HINTS": "HINTS","HINTSProxy": "HINTS Proxy"}
    smc_samplers = []
    mcmc_samplers = []

    # run_experiments.py writes a list of experiments, a single
    # sampler_diagnostics.py sweep the one config it ran.
    config = load_config(timestamp, results_root)
    entries = config.get("experiments") or ([config] if "dataset" in config else [])
    config_of = {stem: entry for entry in entries for stem in config_stems(entry)}
    skipped = []

    for filename in sorted(os.listdir(results_path)):
        if not filename.endswith("_metrics.npz"):
            continue

        # Split from the right: only the last three fields are in a fixed
        # position, so a dataset name with "_" in it still parses.
        stem = filename[:-len("_metrics.npz")]
        entry = config_of.get(stem)
        if entry is None:
            skipped.append(filename)
            continue
        # A run started from a random tree carries a trailing _init<N>, one
        # started from a given tree _init-<name>; a run started from a stump
        # has neither.
        fields, init_tree_size, init_tree_name = stem, 0, None
        head, _, last = stem.rpartition("_")
        if last.startswith("init-") and len(last) > len("init-"):
            fields, init_tree_name = head, last[len("init-"):]
        elif last.startswith("init") and last[len("init"):].isdigit():
            fields, init_tree_size = head, int(last[len("init"):])
        dataset, sampler, proposal, ss = fields.rsplit("_", 3)
        # MH runs on the full data -- None keeps it apart from the subsampled runs.
        ss_prop = None if proposal == "MH" else parse_ss(ss)

        file_path = os.path.join(results_path, filename)
        with np.load(file_path) as npz:
            # Read the arrays out now: np.load is lazy and holds the file open.
            data = {key: npz[key] for key in npz.files}

        sampler_dict = {
            "name": stem,
            "dataset": dataset,
            "ss_prop": ss_prop,
            "init_tree_size": init_tree_size,
            "init_tree_name": init_tree_name,
            "sampler": sampler,
            "proposal_mechanism": PROPOSALS.get(proposal, proposal),
            "path": file_path,
            "h5_path": os.path.join(results_path, stem + ".h5"),
            # Every stride-th stored record was evaluated (80 for smc_MH_100, 1
            # for the rest). data['iterations'] already holds the iteration /
            # step each evaluated point came from, so plot against that rather
            # than against the point's index.
            "stride": int(data["stride"]),
            "data": data,
            # From the matching config.json experiment. An entry may leave any
            # of these out, which reads as None here.
            # init_tree is the starting tree itself, as [node, left, right,
            # feat, thr] rows -- None unless the runs were given one.
            "init_tree": entry.get("init_tree"),
            "chains": entry.get("chains"),
            "particles": entry.get("particles") if sampler == "smc" else None,
            "max_tree_size": entry.get("max_tree_size"),
        }

        if sampler == "smc":
            smc_samplers.append(sampler_dict)
        else:
            mcmc_samplers.append(sampler_dict)

    if skipped:
        print(f"Skipped {len(skipped)} metrics file(s) with no matching experiment "
              f"in config.json: {', '.join(skipped)}")
    return smc_samplers, mcmc_samplers

In [ ]:
timestamp="261113_170316"
smc_results, mcmc_results = load_metrics_files(timestamp)

In [ ]:
smc_results[0].keys()

In [ ]:
# reorder the results to go in the order: DA, Flat HINTS, HINTS, MH and then for each of the first 3, order by ss_prop ascending
def reorder_results(results):
    proposal_order = ["DA", "Flat HINTS", "HINTS", "HINTS Proxy", "MH"]
    results.sort(key=lambda x: (proposal_order.index(x["proposal_mechanism"]), x["ss_prop"] if x["ss_prop"] is not None else float('inf')))
    return results

mcmc_results = reorder_results(mcmc_results)
smc_results = reorder_results(smc_results)

In [ ]:
for experiment in smc_results + mcmc_results:
    if experiment["proposal_mechanism"] == "HINTS" or experiment["proposal_mechanism"] == "HINTS Proxy":
        # The hierarchy design, e.g. ss_prop=0.125 -> [1, 2, 4, 8]: 1/ss_prop
        # blocks at the bottom level, halving until a single block at the top.
        ss_prop = experiment["ss_prop"]
        experiment["ss_prop"] = [2**i for i in range(int(np.log2(round(1 / ss_prop))) + 1)]
    elif experiment["proposal_mechanism"] == "Flat HINTS":
        experiment["ss_prop"] = int(round(1 / experiment["ss_prop"]))


In [ ]:
# Per-iteration runtime lives in the .h5 each metrics file was evaluated from:
# every Run_N group holds iter_time and cumulative_time, one entry per
# iteration (MCMC) or step (SMC). The clock times the sampler alone -- the
# time spent recording states is excluded.

import h5py


def record_positions(experiment):
    """Which stored records were evaluated: every stride-th one. state_index
    is indexed by record, while cumulative_time and ess are indexed by
    iteration -- the two only coincide when stride and store_every are 1, which
    the stride-80 smc_MH_100 run is not."""
    return np.arange(len(experiment['data']['iterations'])) * experiment['stride']


def load_n_iters(experiment):
    """Iterations (MCMC) / steps (SMC) the run actually took."""
    with h5py.File(experiment['h5_path'], "r") as f:
        return len(f["Run_0"]["cumulative_time"])


def load_runtime(experiment):
    """Cumulative sampling time in seconds, as an (n_runs, n_points) array
    lined up with the experiment's metric arrays."""
    h5_path = experiment['h5_path']
    iterations = experiment['data']['iterations']
    n_runs = int(experiment['data']['n_runs'])
    with h5py.File(h5_path, "r") as f:
        runs = sorted((k for k in f if k.startswith("Run_")), key=lambda k: int(k[len("Run_"):]))
        return np.stack([f[run]["cumulative_time"][()][iterations] for run in runs[:n_runs]])

def load_ess(experiment): # only for SMC experiments
    """Effective sample size (ESS) per iteration, as an (n_runs, n_points) array
    lined up with the experiment's metric arrays."""
    h5_path = experiment['h5_path']
    iterations = experiment['data']['iterations']
    n_runs = int(experiment['data']['n_runs'])
    with h5py.File(h5_path, "r") as f:
        runs = sorted((k for k in f if k.startswith("Run_")), key=lambda k: int(k[len("Run_"):]))
        return np.stack([f[run]["ess"][()][iterations] for run in runs[:n_runs]])

def load_tree_len(experiment, particles="mean"):
    """Tree size in nodes, as an (n_runs, n_points) array lined up with the
    experiment's metric arrays.

    state_tree_lengths holds one entry per *stored* tree, not per iteration:
    consecutive identical states are kept once, so a 10k-iteration run may
    store a couple of hundred trees. state_index is what maps an iteration (and,
    for SMC, a particle) onto a slot in that store -- indexing the store with
    the iteration numbers directly reads unrelated trees and runs off the end.

    MCMC carries one tree per iteration; SMC carries one per particle, so
    `particles` says how to reduce them: "mean", "max", or "all" to keep the
    full (n_runs, n_points, n_particles) array. All three agree on MCMC, where
    there is only one particle to reduce over.
    """
    reducers = {"mean": lambda a: a.mean(axis=1),
                "max": lambda a: a.max(axis=1),
                "all": lambda a: a}
    if particles not in reducers:
        raise ValueError(f"particles must be one of {sorted(reducers)}, got {particles!r}")
    reduce = reducers[particles]

    h5_path = experiment['h5_path']
    n_runs = int(experiment['data']['n_runs'])
    with h5py.File(h5_path, "r") as f:
        runs = sorted((k for k in f if k.startswith("Run_")), key=lambda k: int(k[len("Run_"):]))
        return np.stack([
            reduce(f[run]["state_tree_lengths"][()][f[run]["state_index"][()][record_positions(experiment)]])
            for run in runs[:n_runs]])


for experiment in smc_results + mcmc_results:
    experiment['time'] = load_runtime(experiment)
    experiment['tree_len'] = load_tree_len(experiment)
    experiment['n_iters'] = load_n_iters(experiment)

for experiment in smc_results:
    experiment['ess'] = load_ess(experiment)

In [ ]:
# plot mean and std of ess for all SMC experiments, each plot having a different proposal mechanism.
from matplotlib.pyplot import plot


num_plots = len(smc_results)
ncols = 2
nrows = int(np.ceil(num_plots / ncols))
# Change order to be HINTS, Flat HINTS, DA, MH
# window = 2

smc_results_reordered = [e for e in smc_results if e['proposal_mechanism'] == 'HINTS'] + \
                        [e for e in smc_results if e['proposal_mechanism'] == 'HINTS Proxy'] + \
                        [e for e in smc_results if e['proposal_mechanism'] == 'Flat HINTS'] + \
                        [e for e in smc_results if e['proposal_mechanism'] == 'DA'] + \
                        [e for e in smc_results if e['proposal_mechanism'] == 'MH']

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols * 6, nrows * 4),sharey=True)
for i, experiment in enumerate(smc_results_reordered):
    ax = axes[i // ncols, i % ncols]
    # plt.plot(range(window, len(per_iter_time)), [np.mean(per_iter_time[i-window:i]) for i in range(window, len(per_iter_time))])
    pit = [experiment['time'][:, k]-experiment['time'][:, k-1] for k in range(1, experiment['time'].shape[1])]
    pit_mean = [np.mean(experiment['time'][:, k]-experiment['time'][:, k-1]) for k in range(1, experiment['time'].shape[1])]
    pit_std = [np.std(experiment['time'][:, k]-experiment['time'][:, k-1]) for k in range(1, experiment['time'].shape[1])]
    for j in range(len(pit[0])):
        chain = [pit[i][j] for i in range(len(pit))]
        ax.plot(chain,linewidth=0.75,color="tab:blue",alpha=0.2,label = "chain runtime per iter" if j == 0 else None)
    ax.plot(pit_mean,linewidth=1.25,color="blue",alpha=1,label="mean runtime")
    # ax.plot(range(window, len(pit)), [np.mean(pit[i-window:i]) for i in range(window, len(pit))], color="red",linewidth=1,label=f"{window}-iter moving average")
    if "HINTS" in experiment['proposal_mechanism']:
        ax.set_title(f"{experiment['proposal_mechanism']} (Hierarchy={experiment['ss_prop']})")
    elif "MH" in experiment['proposal_mechanism']:
        ax.set_title("Metropolis-Hastings")
    else:
        ax.set_title(f"{experiment['proposal_mechanism']} (ss_prop={experiment['ss_prop']})")
    ax.set_xlabel('Iterations')
    ax.set_ylabel('Run Time (s)')
    ax.set_yscale('log')

plt.tight_layout()
fig.legend(axes[0, 0].get_legend_handles_labels()[0], axes[0, 0].get_legend_handles_labels()[1], loc='upper center', ncol=5, bbox_to_anchor=(0.5, 1.02))
fig.suptitle(f"Run Time Per Iteration for SMC Experiments - {timestamp}", fontsize=16,y=1.035)
plt.show()

In [ ]:
# plot mean and std of ess for all SMC experiments, each plot having a different proposal mechanism.
num_plots = len(smc_results)
ncols = 2
nrows = int(np.ceil(num_plots / ncols))
# Change order to be HINTS, Flat HINTS, DA, MH

smc_results_reordered = [e for e in smc_results if e['proposal_mechanism'] == 'HINTS'] + \
                        [e for e in smc_results if e['proposal_mechanism'] == 'HINTS Proxy'] + \
                        [e for e in smc_results if e['proposal_mechanism'] == 'Flat HINTS'] + \
                        [e for e in smc_results if e['proposal_mechanism'] == 'DA'] + \
                        [e for e in smc_results if e['proposal_mechanism'] == 'MH']

fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols * 6, nrows * 4),sharey=True, sharex=True)
for i, experiment in enumerate(smc_results_reordered):
    ax = axes[i // ncols, i % ncols]
    ess_mean = np.mean(experiment['ess'], axis=0)
    ess_std = np.std(experiment['ess'], axis=0)
    ax.plot(experiment['data']['iterations'], ess_mean, label='Mean ESS',linewidth=1)
    ax.fill_between(experiment['data']['iterations'], [max(0, a) for a in ess_mean - ess_std ], [min(40, a) for a in ess_mean + ess_std ], alpha=0.3, label='Std Dev')
    if "HINTS" in experiment['proposal_mechanism']:
        ax.set_title(f"{experiment['proposal_mechanism']} (Hierarchy={experiment['ss_prop']})")
    elif "MH" in experiment['proposal_mechanism']:
        ax.set_title("Metropolis-Hastings")
    else:
        ax.set_title(f"{experiment['proposal_mechanism']} (ss_prop={experiment['ss_prop']})")
    ax.set_xlabel('Iterations')
    ax.set_ylabel('Effective Sample Size (ESS)')
    ax.axhline(y=experiment['particles']//2, color='red', linestyle='--', label=f'N/2', linewidth=2)
    # ax.legend()

plt.tight_layout()
fig.legend(axes[0, 0].get_legend_handles_labels()[0], axes[0, 0].get_legend_handles_labels()[1], loc='upper center', ncol=5, bbox_to_anchor=(0.5, 1.02))
fig.suptitle(f"Effective Sample Size (ESS) for SMC Experiments - {timestamp}", fontsize=16,y=1.035)
plt.show()

In [ ]:
# plot mean and std of ess for all SMC experiments, each plot having a different proposal mechanism.
num_plots = len(smc_results)-1
ncols = 2
nrows = int(np.ceil(num_plots / ncols))+1
fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols * 6, nrows * 4),sharey=True)
for i, experiment in enumerate(smc_results_reordered):
    ax = axes[i // ncols, i % ncols]
    ess_mean = np.mean(experiment['ess'], axis=0)
    ess_std = np.std(experiment['ess'], axis=0)
    ax.plot(experiment['data']['iterations'], ess_mean, label='Mean ESS',linewidth=1)
    ax.fill_between(experiment['data']['iterations'], [max(0, a) for a in ess_mean - ess_std ], [min(40, a) for a in ess_mean + ess_std ], alpha=0.3, label='Std Dev')
    if "HINTS" in experiment['proposal_mechanism']:
        ax.set_title(f"{experiment['proposal_mechanism']} (Hierarchy={experiment['ss_prop']})")
    elif "MH" in experiment['proposal_mechanism']:
        ax.set_title("Metropolis-Hastings")
    else:
        ax.set_title(f"{experiment['proposal_mechanism']} (ss_prop={experiment['ss_prop']})")
    ax.set_xlabel('Iterations')
    ax.set_ylabel('Effective Sample Size (ESS)')
    ax.axhline(y=experiment['particles']//2, color='red', linestyle='--', label=f'N/2', linewidth=2)
    ax.set_xlim([-1, 1+max([e['data']['iterations'][-1] for e in smc_results_reordered if e['proposal_mechanism'] == 'HINTS'])])
    # ax.legend()

plt.tight_layout()
fig.legend(axes[0, 0].get_legend_handles_labels()[0], axes[0, 0].get_legend_handles_labels()[1], loc='upper center', ncol=5, bbox_to_anchor=(0.5, 1.02))
fig.suptitle(f"Effective Sample Size (ESS) for SMC Experiments - {timestamp} for {1+max([e['data']['iterations'][-1] for e in smc_results_reordered if e['proposal_mechanism'] == 'HINTS'])} iterations", fontsize=16,y=1.035)
plt.show()

In [ ]:


# reverse order of DA experiments so it matches Flat HINTS and HINTS, which are ordered by ss_prop ascending
def reorder_results(results):
    # reverse DA 
    order = ["DA", "Flat HINTS", "HINTS Proxy", "HINTS", "MH"]
    ss_order = {proposal: sorted([expe for expe in results if expe['proposal_mechanism'] == proposal], key=lambda x: x['ss_prop'] if x['ss_prop'] is not None else float('inf')) for proposal in order}
    ss_reversed = {proposal: list(reversed(ss_order[proposal])) for proposal in order if proposal == "DA"}
    for proposal in order:
        if proposal == "DA":
            results = [expe for expe in results if expe['proposal_mechanism'] != proposal] + ss_reversed[proposal]
        else:
            results = [expe for expe in results if expe['proposal_mechanism'] != proposal] + ss_order[proposal]
    return results
smc_combined = reorder_results(smc_results)
mcmc_combined = reorder_results(mcmc_results)

In [ ]:
line_colours = {
    'MH':    ['#009E73'],
    'DA':    ['#8B0A2E', '#D55E00', '#E69F00', '#F0E442'],
    'HINTS2': ['#56B4E9', '#0072B2', '#2B2B7F', '#7B3294']}
shade_colours = {
    'MH':    ['#99D8C7'],
    'DA':    ['#E3B5C1', '#F2BF99', '#F5D999', '#FAF4B3'],
    'HINTS2': ['#BBE1F6', '#99C7E0', '#B9B9DC', '#C4A3CF']}

line_colours = {
    'MH':    ['#009E73'],
    # 'DA':    ['#CC79A7', '#0072B2', '#F0E442', '#56B4E9', '#E69F00'],
    'DA':    ['#E69F00', '#56B4E9','#CC79A7', '#F0E442', '#0072B2'],
    'HINTS': ['#E69F00', '#56B4E9','#CC79A7', '#F0E442', '#0072B2'],
    'HINTS Proxy': ['#E69F00', '#56B4E9','#CC79A7', '#F0E442', '#0072B2'],
    'Flat HINTS': ['#E69F00', '#56B4E9','#CC79A7', '#F0E442', '#0072B2']
}
shade_colours = {
    'MH':    ['#99D8C7'],
    # 'DA':    ['#D9A3D9', '#99C7E0', '#FAF4B3', '#BBE1F6', '#F2BF99'],
    'DA':   ['#F2BF99', '#BBE1F6','#D9A3D9', '#FAF4B3', '#99C7E0'],
    'HINTS': ['#F2BF99', '#BBE1F6','#D9A3D9', '#FAF4B3', '#99C7E0'],
    'HINTS Proxy': ['#F2BF99', '#BBE1F6','#D9A3D9', '#FAF4B3', '#99C7E0'],
    'Flat HINTS': ['#F2BF99', '#BBE1F6','#D9A3D9', '#FAF4B3', '#99C7E0']
}
X_LABELS = {"iteration": "Iteration", "time": "Sampling time (s)"}

def colour_index(experiment, results):
    """Rank of this experiment's ss_prop among its proposal's ss_props, so each
    subsample size keeps the same colour in every plot."""
    proposal = experiment['proposal_mechanism']
    if isinstance(experiment['ss_prop'], list):
        ss_value = experiment['ss_prop'][-1]
    else:
        ss_value = experiment['ss_prop']
    ss_values = sorted({e['ss_prop'][-1] if isinstance(e['ss_prop'], list) else e['ss_prop'] for e in results if e['proposal_mechanism'] == proposal},
                       key=lambda s: -1 if s is None else s)
    return ss_values.index(ss_value) % len(line_colours[proposal])


def curve(experiment, key, x="iteration", n_grid=500):
    """(x values, mean, std) of a metric across runs, against iteration or,
    with x="time", against cumulative sampling time in seconds.

    `key` is looked up in experiment['data'] (the evaluated metrics) and then
    on the experiment itself, which is where the series loaded from the .h5
    live: 'tree_len', and 'ess' for SMC."""
    data = experiment['data']
    values = data[key] if key in data else experiment[key]   # (n_runs, n_points)
    if x == "iteration":
        return data['iterations'], values.mean(axis=0), values.std(axis=0)
    # Runs finish at different wallclock times, so they are averaged on a shared
    # time grid that stops when the fastest run finished; at each grid time a
    # run holds the value of the last iteration it had completed.
    times = experiment['time']
    grid = np.linspace(0.0, times[:, -1].min(), n_grid)
    at_grid = np.stack([v[np.clip(np.searchsorted(t, grid, side="right") - 1, 0, None)]
                        for t, v in zip(times, values)])
    return grid, at_grid.mean(axis=0), at_grid.std(axis=0)



# same as above but separate HINTS2 and DA into 2 subplots with MH on both
def plot_metric_split(results, key, title, samplername, ylabel, cols=4, x="iteration",footnote=None, xlim=None, yscale=None, sharex=True,sharey=True, ignore_mh=False):
    fig, axes = plt.subplots(1, cols, figsize=(20, 6), sharey=sharey, sharex=sharex)
    if key == "ess":
        neff = results[0]['particles']//2
    for experiment in results:
        proposal = experiment['proposal_mechanism']
        max_tree_size = experiment['max_tree_size']
        k = colour_index(experiment, results)
        xs, mean, std = curve(experiment, key, x)
        if experiment['proposal_mechanism'] == "HINTS":
            label = f"HINTS (Hierarchy={experiment['ss_prop']})"
        elif experiment['proposal_mechanism'] == "HINTS Proxy":
            label = f"HINTS Proxy (Hierarchy={experiment['ss_prop']})"
        elif experiment['proposal_mechanism'] == "Flat HINTS":
            label = f"Flat HINTS (Hierarchy={experiment['ss_prop']})"
        elif experiment['proposal_mechanism'] == "DA":
            label = f"DA (ss={experiment['ss_prop']})"
        else:
            label = proposal
        # label = proposal if experiment['ss_prop'] is None else f"ss={experiment['ss_prop']}"
        if proposal == "DA":
            ax = axes[0]
        elif proposal == "Flat HINTS":
            ax = axes[1]
        elif proposal == "HINTS Proxy":
            ax = axes[2]
        elif proposal == "HINTS":
            ax = axes[3]
        elif proposal == "MH":
            for ax in axes:
                if key == "tree_len":
                    ax.fill_between(xs, [max(0, a) for a in  mean - std], [min(max_tree_size, a) for a in mean + std ], color=shade_colours[proposal][k], alpha=0.5, zorder=1)
                else:
                    ax.fill_between(xs, mean - std, mean + std, color=shade_colours[proposal][k], alpha=0.5, zorder=1)
                ax.plot(xs, mean, label=label, color=line_colours[proposal][k], zorder=2)
                if key == "ess":
                    ax.axhline(y=neff, color='red', linestyle='--', label=f'N/2', linewidth=2)
        if proposal in ["DA","Flat HINTS","HINTS Proxy","HINTS"]:
            if key == "tree_len":
                ax.fill_between(xs, [max(0, a) for a in  mean - std], [min(max_tree_size, a) for a in mean + std ], color=shade_colours[proposal][k], alpha=0.5, zorder=1)
            else:
                ax.fill_between(xs, mean - std, mean + std, color=shade_colours[proposal][k], alpha=0.5, zorder=1)
            ax.plot(xs, mean, label=label, color=line_colours[proposal][k], zorder=2)
            ax.set_title(f"{proposal}")
            ax.set_xlabel(X_LABELS[x])
        if xlim:
            ax.set_xlim(xlim)
        if yscale:
            ax.set_yscale(yscale)
        ax.set_ylabel(ylabel)
        ax.legend()
    if xlim:
        fig.suptitle(f"{samplername} {title} by {x} on {experiment['dataset']} for {experiment['data']['n_runs']} runs (First {max(xlim)} {"seconds" if x=="time" else "iterations"})")
    else:
        fig.suptitle(f"{samplername} {title} by {x} on {experiment['dataset']} for {experiment['data']['n_runs']} runs")
    # add footnote if provided
    if footnote:
        plt.figtext(0.5, -0.05, footnote, wrap=True, horizontalalignment='center', fontsize=10, fontstyle='italic')
    plt.show()


def mh_footnote(results, unit):
    """How MH's run differs from the rest, read off the files themselves."""
    mh = [e for e in results if e['proposal_mechanism'] == 'MH'][0]
    rest = [e for e in results if e['proposal_mechanism'] == 'HINTS'][0]
    note = (f"MH and DA were run for {mh['n_iters'] / rest['n_iters']:g} times as many "
            f"{unit}s as the other proposal mechanisms")
    if mh['stride'] > 1:
        note += (f", with its metrics evaluated every {mh['stride']}nd {unit} "
                 f"and plotted at those {unit}s")
    return note + "."

def plot_chains(results, key, samplername, label, x="iteration", footnote=None,xlim=None):
    nplots = len(results)
    ncols = len([exp for exp in results if exp['proposal_mechanism'] == "HINTS"])
    nrows = int(np.ceil(nplots / ncols))
    fig, axes = plt.subplots(nrows=nrows, ncols=ncols, figsize=(ncols * 6, nrows * 4),sharey=True)
    for i, experiment in enumerate(results):
        data = experiment['data']
        if nrows == 1 or ncols == 1:
            ax = axes[i % max(nrows, ncols)]
        else:
            ax = axes[i // ncols, i % ncols]
        if x == "iteration":
            x_label = "Iterations"
            title_label = "iteration"
            title_unit = "iterations"
            iters = data['iterations']
        else:
            chain_iters = experiment['time']
            iters = np.mean(chain_iters, axis=0)
            x_label = "Time (s)"
            title_label = "second"
            title_unit = "seconds"
        acc = data[key] if key in data else experiment[key]
        mean = np.mean(acc,axis=0)
        std = np.std(acc,axis=0)
        # max_acc = np.max(acc,axis=0)
        # min_acc = np.min(acc,axis=0)
        if x == "iteration":
            for j, chain in enumerate(acc):
                ax.plot(iters,chain,linewidth=1,color="blue",alpha=0.25,label = f"chain" if j == 0 else None)
        else:
            for j, chain in enumerate(acc):
                ax.plot(chain_iters[j],chain,linewidth=1,color="blue",alpha=0.25,label = f"chain" if j == 0 else None)
        ax.plot(iters,mean,linewidth=2,color="red",alpha=1,label="mean")
        ax.plot(iters,mean+std,linewidth=2,color="orange",linestyle="--",alpha=1,label="+1 std")
        ax.plot(iters,mean-std,linewidth=2,color="orange",linestyle="--",alpha=1,label="-1 std")
        if "HINTS" in experiment['proposal_mechanism']:
            ax.set_title(f"{experiment['proposal_mechanism']} (Hierarchy={experiment['ss_prop']})")
        elif "MH" in experiment['proposal_mechanism']:
            ax.set_title("Metropolis-Hastings")
        else:
            ax.set_title(f"{experiment['proposal_mechanism']} (ss_prop={experiment['ss_prop']})")
        ax.set_xlabel(x_label)
        ax.set_ylabel(label)
        if xlim is not None:
            ax.set_xlim(xlim)
    plt.tight_layout()
    handles, labels = [], []
    for ax in axes.flatten():
        h, l = ax.get_legend_handles_labels()
        handles.extend(h)
        labels.extend(l)
    handles, labels = zip(*dict(zip(labels, handles)).items())  # remove duplicates
    fig.legend(handles, labels, loc='upper center', ncol=5, bbox_to_anchor=(0.5, 1.02))
    if xlim is not None:
        fig.suptitle(f"{label} Per {title_label} for {samplername} Experiments ({xlim[0]}-{xlim[1]} {title_unit})  - {timestamp}", fontsize=16,y=1.035)
    else:
        fig.suptitle(f"{label} Per {title_label} for {samplername} Experiments - {timestamp}", fontsize=16,y=1.035)
    if footnote:
        plt.figtext(0.5, -0.05, footnote, wrap=True, horizontalalignment='center', fontsize=10, fontstyle='italic')
    plt.show()

# SMC RESULTS

In [ ]:
footnote = mh_footnote(smc_combined, "step")

In [ ]:
plot_metric_split(smc_combined, 'tree_len', "Tree Size", "SMC", "Internal nodes", footnote=footnote)
plot_metric_split(smc_combined, 'tree_len', "Tree Size", "SMC", "Internal nodes", x="time", footnote=footnote)

In [ ]:
plot_metric_split(smc_combined, 'train_accuracy', "Training Accuracy", "SMC", "Accuracy",footnote=footnote)
plot_metric_split(smc_combined, 'train_accuracy', "Training Accuracy", "SMC", "Accuracy",x="time",footnote=footnote)

In [ ]:
plot_metric_split(smc_combined, 'test_accuracy', "Test Accuracy", "SMC", "Accuracy",footnote=footnote, xlim=[-1,50])
plot_metric_split(smc_combined, 'test_accuracy', "Test Accuracy", "SMC", "Accuracy", x="time",footnote=footnote, xlim=[-1,15])


In [ ]:
plot_metric_split(smc_combined, 'test_brier', "Test Brier Score", "SMC", "Brier Score",footnote=footnote)
plot_metric_split(smc_combined, 'test_brier', "Test Brier Score", "SMC", "Brier Score", x="time",footnote=footnote)

In [ ]:
plot_metric_split(smc_combined, 'test_macro_f1', "Test Macro F1", "SMC", "Macro F1 Score",footnote=footnote)
plot_metric_split(smc_combined, 'test_macro_f1', "Test Macro F1", "SMC", "Macro F1 Score", x="time",footnote=footnote)


In [ ]:
plot_metric_split(smc_combined, 'test_log_loss', "Test Log Loss", "SMC", "Log Loss",footnote=footnote)
plot_metric_split(smc_combined, 'test_log_loss', "Test Log Loss", "SMC", "Log Loss", x="time",footnote=footnote)

# MCMC RESULTS

In [ ]:
footnote = mh_footnote(mcmc_combined, "iteration")


In [ ]:
plot_chains(mcmc_results, "tree_len", "MCMC", "Tree Size", x="iteration", footnote=footnote)

In [ ]:
plot_chains(mcmc_results, "test_accuracy", "MCMC", "Test Accuracy", x="iteration", footnote=footnote)

In [ ]:
plot_metric_split(smc_combined, 'tree_len', "Tree Size", "SMC", "Internal nodes", footnote=footnote)
plot_metric_split(smc_combined, 'tree_len', "Tree Size", "SMC", "Internal nodes", x="time", footnote=footnote)

In [ ]:
plot_metric_split(mcmc_combined, 'train_accuracy', "Training Accuracy", "MCMC", "Accuracy",footnote=footnote)
plot_metric_split(mcmc_combined, 'train_accuracy', "Training Accuracy", "MCMC", "Accuracy", x="time",footnote=footnote)


In [ ]:
plot_metric_split(mcmc_combined, 'test_accuracy', "Test Accuracy", "MCMC", "Accuracy",footnote=footnote)
plot_metric_split(mcmc_combined, 'test_accuracy', "Test Accuracy", "MCMC", "Accuracy", x="time",footnote=footnote)

In [ ]:
plot_metric_split(mcmc_combined, 'test_log_loss', "Test Log Loss", "MCMC", "Log Loss",footnote=footnote)
plot_metric_split(mcmc_combined, 'test_log_loss', "Test Log Loss", "MCMC", "Log Loss", x="time",footnote=footnote)


In [ ]:
# rearrange experiments in smc_combined and mcmc_combined so that MH is last
smc_combined = sorted(smc_combined, key=lambda x: (x['proposal_mechanism'] == 'MH', x['proposal_mechanism'], x['ss_prop']))
mcmc_combined = sorted(mcmc_combined, key=lambda x: (x['proposal_mechanism'] == 'MH', x['proposal_mechanism'], x['ss_prop']))

In [ ]:
# From the h5 files, the fate of every proposed move, broken down by move type.
# Each Run_N holds "move" (0-3, MOVES) and "outcome" (0-5, OUTCOMES).
from matplotlib.patches import Patch


MOVES = ("stay", "grow", "prune", "change")
MOVE_CODE = {name: i for i, name in enumerate(MOVES)}

STAY = 0            # no valid move was available
BARRED = 1          # the proposal correction was BARRED (invalid split, or a reverse path barred at max_tree_size)
SCREENED_OUT = 2    # rejected by the surrogate on a subsample (DA, HINTS)
INADMISSIBLE = 3    # rejected by a min_samples_leaf admissibility check
PROPOSED = 4        # survived the proposal, handed to the outer MH step
APPLIED = 5         # written into a HINTS sweep's working tree
OUTCOMES = ("stay", "barred", "screened_out", "inadmissible", "proposed",
            "applied")

# Segments stack in outcome order, so the hues were picked to clear the
# adjacent-pair colour-blindness and contrast checks in that order -- the
# saturated red/yellow/green they replace failed both, and pure yellow on a
# white ground was invisible.
OUTCOME_COLOURS = {
    STAY:         "#8c8578",  # warm grey -- the null outcome
    BARRED:       "#e27860",  # orange-red
    SCREENED_OUT: "#a59ae6",  # violet
    INADMISSIBLE: "#e9de65",  # amber
    PROPOSED:     "#8bb4e5",  # blue
    APPLIED:      "#6AE26A",  # green
}

SURFACE = "#ffffff"
BAR_W = 0.5
INSIDE_MIN = 0.055   # segments thinner than this fraction of the axis get a lifted label
GAP = 0.052          # vertical spacing of the lifted labels, same units


def move_outcome_counts(experiment):
    """(len(MOVES), len(OUTCOMES)) counts, pooled over every chain of a run."""
    h5_path = experiment['h5_path']
    with h5py.File(h5_path, "r") as f:
        runs = sorted((k for k in f if k.startswith("Run_")),
                      key=lambda k: int(k[len("Run_"):]))
        moves = np.concatenate([f[run]["move"][()] for run in runs])
        outcomes = np.concatenate([f[run]["outcome"][()] for run in runs])
    flat = moves.astype(np.int64) * len(OUTCOMES) + outcomes.astype(np.int64)
    return np.bincount(flat, minlength=len(MOVES) * len(OUTCOMES)).reshape(
        len(MOVES), len(OUTCOMES))


def _si(n):
    for cut, suffix in ((1e9, "B"), (1e6, "M"), (1e3, "k")):
        if n >= cut:
            return f"{n / cut:.1f}{suffix}".replace(".0", "")
    return str(int(n))


def _pct(value):
    return f"{value:.2f}%" if value >= 1 else f"{value:.3f}%"


def _n_lifted(heights, span):
    """How many segments are too thin to hold their own label."""
    return sum(1 for h in heights if 0 < h < INSIDE_MIN * span)


def _label_bar(ax, i, heights, shares, span):
    """Percentages beside the segments they belong to, each in that outcome's
    colour over a black outline -- the outline is what keeps a label legible
    where it sits on its own fill, or on the white ground above the bar."""
    # A thin outline: a heavy one swallows the glyph at this size, which is
    # what makes a label unreadable when it sits on a fill of its own colour.
    stroke = [pe.withStroke(linewidth=1.5, foreground="black")]
    gap = GAP * span
    y = heights.sum() + gap
    j = 0
    for k in range(len(OUTCOMES)):
        if heights[k] == 0:
            continue
        centre = heights[:k].sum() + heights[k] / 2
        if heights[k] >= INSIDE_MIN * span:
            ax.text(i, centre, _pct(shares[k]), ha='center', va='center',
                    color=OUTCOME_COLOURS[k], size=10, weight='bold',
                    zorder=4, path_effects=stroke)
        else:
            # Out of the segment sideways, then straight up to the label, so a
            # leader never cuts across the stack or a neighbouring bar.
            ax.annotate(_pct(shares[k]), xy=(i + BAR_W / 2, centre),
                        xytext=(i + BAR_W / 2 + 0.05 + 0.055 * j, y),
                        ha='left', va='center', size=10, weight='bold', zorder=4,
                        color=OUTCOME_COLOURS[k], path_effects=stroke,
                        annotation_clip=False,
                        arrowprops=dict(arrowstyle='-', lw=0.8, shrinkA=0, shrinkB=2,
                                        color=OUTCOME_COLOURS[k],
                                        connectionstyle="angle,angleA=0,angleB=90"))
            y += gap
            j += 1


def plot_move_outcomes(results, samplername, share=True, footnote=None):
    """One panel per experiment: proposal outcomes stacked by move type.

    share=True stacks every bar to 100% and prints its total underneath -- the
    totals run over orders of magnitude between moves and between experiments,
    so raw counts flatten every bar but the largest. share=False plots the
    counts themselves, each panel on its own y axis.
    """
    counts = [move_outcome_counts(experiment) for experiment in results]

    ncols = 3
    nrows = int(np.ceil(len(results) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5.4, nrows * 4.4),
                             squeeze=False)
    axes = axes.ravel()
    xs = np.arange(len(MOVES))

    panels = []
    for experiment, count in zip(results, counts):
        totals = count.sum(axis=1)
        shares = np.divide(count * 100.0, totals[:, None], out=np.zeros(count.shape),
                           where=totals[:, None] > 0)
        heights = shares if share else count.astype(float)
        span = 100.0 if share else max(heights.sum(axis=1).max(), 1.0)
        # Headroom for the tallest column of lifted labels, so none is drawn
        # outside the axes -- tight_layout would squash the panel to fit it.
        n_lift = max(_n_lifted(heights[i], span) for i in range(len(MOVES)))
        panels.append((experiment, totals, shares, heights, span,
                       span * (1 + GAP * (n_lift + 0.8))))

    # A shared limit only means anything when every panel is on the same 0-100 scale.
    shared_top = max(p[-1] for p in panels) if share else None

    for ax, (experiment, totals, shares, heights, span, top) in zip(axes, panels):
        bottom = np.zeros(len(MOVES))
        for k in range(len(OUTCOMES)):
            ax.bar(xs, heights[:, k], width=BAR_W, bottom=bottom, zorder=2,
                   color=OUTCOME_COLOURS[k], edgecolor=SURFACE, linewidth=1.5)
            bottom += heights[:, k]

        for i in range(len(MOVES)):
            if totals[i] > 0:
                _label_bar(ax, i, heights[i], shares[i], span)

        ss = experiment['ss_prop']
        if experiment['proposal_mechanism'] == "HINTS":
            ax.set_title(f"{experiment['proposal_mechanism']} (Hierarchy={ss}, {totals.sum():,} proposals)", fontsize=14, fontweight='bold')
        elif experiment['proposal_mechanism'] == "Flat HINTS":
            ax.set_title(f"{experiment['proposal_mechanism']} (Hierarchy={ss}, {totals.sum():,} proposals)", fontsize=14, fontweight='bold')
        elif experiment['proposal_mechanism'] == "DA":
            ax.set_title(f"{experiment['proposal_mechanism']} (ss_prop={ss}, {totals.sum():,} proposals)", fontsize=14, fontweight='bold')
        else:
            ax.set_title(f"{experiment['proposal_mechanism']} (ss_prop={ss}, {totals.sum():,} proposals)", fontsize=14, fontweight='bold')
        # Under each move: how many proposals it drew, and what share of all
        # the run's proposals that is -- the stack above is a share of n alone.
        proposed_share = 100.0 * totals / max(totals.sum(), 1)
        ax.set_xticks(xs, [f"{m}\nn={_si(t)}\n{_pct(s)}"
                           for m, t, s in zip(MOVES, totals, proposed_share)])
        ax.set_xlim(-0.55, len(MOVES) - 0.15)
        ax.set_ylim(0, shared_top if share else top)
        ax.set_xlabel('Move type')
        ax.set_ylabel('Share of proposals (%)' if share else 'Proposals')
        ax.grid(axis='x', visible=False)
        if share:
            ax.set_yticks(np.arange(0, 101, 25))

    for ax in axes[len(results):]:      # blank out the unfilled grid slots
        ax.set_visible(False)

    fig.legend(handles=[Patch(facecolor=OUTCOME_COLOURS[k], label=OUTCOMES[k])
                        for k in range(len(OUTCOMES))],
               loc='upper center', ncol=len(OUTCOMES), frameon=False,
               bbox_to_anchor=(0.5, 0.97))
    fig.suptitle(f"Proposal outcome by move type -- {samplername} experiments, {timestamp}",
                 fontsize=16)
    if footnote:
        fig.text(0.5, -0.01, footnote, ha='center', fontsize=10, fontstyle='italic')
    fig.tight_layout(rect=(0, 0.04, 1, 0.98))
    plt.show()

# smc_combined = [exp for exp in smc_results if exp['proposal_mechanism'] != "MH"] + long_smc_mh
# mcmc_combined = [exp for exp in mcmc_results if exp['proposal_mechanism'] != "MH"] + long_mcmc_mh
plot_move_outcomes(smc_combined, "SMC", share=False)
plot_move_outcomes(mcmc_combined, "MCMC")

In [ ]:
# The screen crossed with the sampler's own accept/reject. The outcome column
# alone is only half the story: DA and HINTS screen a move against a surrogate
# before the outer Metropolis step ever sees it, so a move can pass the screen
# and still be thrown away -- or, inside a HINTS sweep, be thrown out by the
# screen while the sweep it belonged to lands. move_fates() does that join on
# the call index; it lives in the experiment scripts, so reuse it rather than
# restate it here.
import sys

_diag_dir = os.path.join(os.getcwd(), "examples", "incremental_decision_tree")
if _diag_dir not in sys.path:
    sys.path.insert(0, _diag_dir)
import plot_diagnostics as _pd

# _pd.FATES in order, under the names we read them by. The last two are not
# fates of a proposed move at all -- they never reached the target -- but they
# are most of the work on some move types, so dropping them would renormalise
# the bars onto a fraction of what the proposal actually did.
FATE_LABELS = ("Approx Accept + Accept", "MH Accept", "Approx Accept + Reject",
               "Approx Reject + Accept", "Approx Reject", "MH Reject",
               "barred / inadmissible", "no move available")

FATE_COLOURS = {
    0: "#98df8a",  # light green -- passed the screen and the sampler took it
    1: "#22A822",  # dark green    -- accepted with no screen (MH, DA's skip path)
    2: "#ff7f0e",  # orange  -- screen passed it, the sampler did not
    3: "#54c8e2",  # violet  -- HINTS: the sweep landed, this move did not
    4: "#ff9896",  # amber   -- the surrogate threw it out
    5: "#e34948",  # red     -- rejected, never screened
    6: "#8c8578",  # warm grey
    7: "#c7c2b8",  # light grey
}


# The coarse view: what actually became of the move. "Accept" is _pd's own
# acceptance numerator, so a move screened out of a HINTS sweep that landed
# counts as a Reject -- the sweep was accepted, that move was not.
FATE_GROUPS = (
    ("Accept",      "#22A822", _pd.ACCEPTED_FATES),
    ("Reject",      "#e34948", tuple(k for k in range(6)
                                     if k not in _pd.ACCEPTED_FATES)),
    ("Barred",      "#8c8578", (6,)),
    ("unavailable", "#c7c2b8", (7,)),
)


def group_fates(count):
    """Collapse a (len(MOVES), len(FATE_LABELS)) table onto FATE_GROUPS."""
    return np.stack([count[:, list(cols)].sum(axis=1)
                     for _, _, cols in FATE_GROUPS], axis=1)


def move_fate_counts(experiment):
    """(len(MOVES), len(FATE_LABELS)) counts, pooled over every chain."""
    h5_path = experiment['h5_path']
    with h5py.File(h5_path, "r") as f:
        runs = sorted((k for k in f if k.startswith("Run_")),
                      key=lambda k: int(k[len("Run_"):]))
        tables = [_pd.move_fates(f[run]) for run in runs]
    return sum(t for t in tables if t is not None)


def _label_bar_fate(ax, i, heights, shares, span, colours):
    """As _label_bar above, but over an arbitrary colour map."""
    stroke = [pe.withStroke(linewidth=1.4, foreground="black")]
    gap = GAP * span
    y = heights.sum() + gap
    j = 0
    for k in range(len(heights)):
        if heights[k] == 0:
            continue
        centre = heights[:k].sum() + heights[k] / 2
        if heights[k] >= INSIDE_MIN * span:
            ax.text(i, centre, _pct(shares[k]), ha='center', va='center',
                    color=colours[k], size=10.5, weight='bold', zorder=4,
                    path_effects=stroke)
        else:
            ax.annotate(_pct(shares[k]), xy=(i + BAR_W / 2, centre),
                        xytext=(i + BAR_W / 2 + 0.05 + 0.055 * j, y),
                        ha='left', va='center', size=9, weight='bold', zorder=4,
                        color=colours[k], path_effects=stroke,
                        annotation_clip=False,
                        arrowprops=dict(arrowstyle='-', lw=0.8, shrinkA=0, shrinkB=2,
                                        color=colours[k],
                                        connectionstyle="angle,angleA=0,angleB=90"))
            y += gap
            j += 1


def plot_move_fates(results, samplername, share=True, footnote=None, ncols=3,
                    grouped=False):
    """One panel per experiment: where each move type's proposals ended up.

    Same layout as plot_move_outcomes, over the screen-crossed-with-accept
    classification instead of the proposal-internal outcome.

    grouped=True collapses the six fates onto Accept / Reject, keeping barred
    and unavailable separate -- neither ever reached the outer step, so folding
    them into Reject would read as the sampler having turned them down.
    """
    counts = [move_fate_counts(experiment) for experiment in results]
    if grouped:
        counts = [group_fates(count) for count in counts]
        labels = [name for name, _, _ in FATE_GROUPS]
        colours = {k: hexcode for k, (_, hexcode, _) in enumerate(FATE_GROUPS)}
    else:
        labels, colours = list(FATE_LABELS), FATE_COLOURS
    n_fates = len(labels)

    nrows = int(np.ceil(len(results) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5.4, nrows * 4.4),
                             squeeze=False)
    axes = axes.ravel()
    xs = np.arange(len(MOVES))

    panels = []
    for experiment, count in zip(results, counts):
        totals = count.sum(axis=1)
        shares = np.divide(count * 100.0, totals[:, None], out=np.zeros(count.shape),
                           where=totals[:, None] > 0)
        heights = shares if share else count.astype(float)
        span = 100.0 if share else max(heights.sum(axis=1).max(), 1.0)
        n_lift = max(sum(1 for h in heights[i] if 0 < h < INSIDE_MIN * span)
                     for i in range(len(MOVES)))
        panels.append((experiment, totals, shares, heights, span,
                       span * (1 + GAP * (n_lift + 0.8))))

    shared_top = max(p[-1] for p in panels) if share else None

    for ax, (experiment, totals, shares, heights, span, top) in zip(axes, panels):
        bottom = np.zeros(len(MOVES))
        for k in range(n_fates):
            ax.bar(xs, heights[:, k], width=BAR_W, bottom=bottom, zorder=2,
                   color=colours[k], edgecolor=SURFACE, linewidth=1.5)
            bottom += heights[:, k]

        for i in range(len(MOVES)):
            if totals[i] > 0:
                _label_bar_fate(ax, i, heights[i], shares[i], span, colours)

        ss = experiment['ss_prop']
        if experiment['proposal_mechanism'] == "HINTS":
            ax.set_title(f"{experiment['proposal_mechanism']} (Hierarchy={ss}, {totals.sum():,} proposals)", fontsize=14, fontweight='bold')
        elif experiment['proposal_mechanism'] == "Flat HINTS":
            ax.set_title(f"{experiment['proposal_mechanism']} (Hierarchy={ss}, {totals.sum():,} proposals)", fontsize=14, fontweight='bold')
        elif experiment['proposal_mechanism'] == "DA":
            ax.set_title(f"{experiment['proposal_mechanism']} (ss_prop={ss}, {totals.sum():,} proposals)", fontsize=14, fontweight='bold')
        else:
            ax.set_title(f"{experiment['proposal_mechanism']} (ss_prop={ss}, {totals.sum():,} proposals)", fontsize=14, fontweight='bold')
        move_share = 100.0 * totals / max(totals.sum(), 1)
        ax.set_xticks(xs, [f"{m}\n{_si(t)}\n{_pct(s)}"
                           for m, t, s in zip(MOVES, totals, move_share)])
        ax.set_xlim(-0.55, len(MOVES) - 0.15)
        ax.set_ylim(0, shared_top if share else top)
        ax.set_xlabel('Move type')
        ax.set_ylabel('Share of proposals (%)' if share else 'Proposals')
        ax.grid(axis='x', visible=False)
        if share:
            ax.set_yticks(np.arange(0, 101, 25))

    for ax in axes[len(results):]:
        ax.set_visible(False)

    # if not first column, remove y axis labels
    for ax in axes:
        if ax.get_subplotspec().colspan.start > 0:
            ax.set_ylabel('')
            ax.set_yticklabels([])

    fig.legend(handles=[Patch(facecolor=colours[k], label=labels[k])
                        for k in range(n_fates)],
               loc='upper center', ncol=4, frameon=False, bbox_to_anchor=(0.5, 0.985))
    fig.suptitle(f"Move Outcomes over {experiment['data']['n_runs']} chains in {samplername} experiments on {experiment['dataset']}",
                 fontsize=20,y=1, fontweight='bold')
    if footnote:
        fig.text(0.5, -0.01, footnote, ha='center', fontsize=10, fontstyle='italic')
    fig.tight_layout(rect=(0, 0.04, 1, 0.98))
    plt.show()


plot_move_fates(mcmc_combined, "MCMC")
plot_move_fates(mcmc_combined, "MCMC", grouped=True)
# SMC reweights particles instead of rejecting them, so every call that
# proposed something became the new particle: the two "Reject" fates are
# structurally empty there, and "Accept" means "survived the proposal".
plot_move_fates(smc_combined, "SMC",
                footnote="SMC has no accept/reject step -- it reweights, so nothing "
                         "lands in the Reject fates and Accept means the proposal survived.")
plot_move_fates(smc_combined, "SMC", grouped=True,
                footnote="SMC has no accept/reject step -- it reweights, so nothing "
                         "lands in Reject and Accept means the proposal survived.")

In [ ]:
plot_move_fates(mcmc_combined, "MCMC", grouped=True)
plot_move_fates(smc_combined, "SMC", grouped=True,
                footnote="SMC has no accept/reject step -- it reweights, so nothing "
                         "lands in Reject and Accept means the proposal survived.")


In [ ]:
group_fates(move_fate_counts(smc_combined[4]))

In [ ]:
import pandas as pd
# give me the raw count of each fate for each move type, for each experiment, in a table with percentages in brackets
def move_fate_table(results, samplername, grouped=False):
    counts = [move_fate_counts(experiment) for experiment in results]
    if grouped:
        counts = [group_fates(count) for count in counts]
        labels = [name for name, _, _ in FATE_GROUPS]
    else:
        labels = list(FATE_LABELS)
    n_fates = len(labels)

    table = []
    for experiment, count in zip(results, counts):
        totals = count.sum(axis=1)
        shares = np.divide(count , totals[:, None], out=np.zeros(count.shape),
                           where=totals[:, None] > 0)
        row = {
            "proposal_mechanism": experiment['proposal_mechanism']
        }
        if experiment['proposal_mechanism'] in ["HINTS", "Flat HINTS"]:
            row["proposal_mechanism"] = f"\\makecell{{{experiment['proposal_mechanism']} \\\\ ${experiment['ss_prop']}$}}"
        elif experiment['proposal_mechanism'] == "DA":
            row["proposal_mechanism"] = f"\\makecell{{{experiment['proposal_mechanism']} \\\\ {float(experiment['ss_prop']):.3g}}}"
        else:
            row["proposal_mechanism"] = "\\makecell{MH}"
        for i, move in enumerate(MOVES):
            for j, fate in enumerate(labels):
                if count[i, j] > 0:
                    # add latex newline between count and percentage
                    row[f"{move}_{fate}"] = f"\\makecell{{{count[i, j]:,} \\\\ ({shares[i, j]:.2g})}}"
        table.append(row)
    return table

mcmc_counts = move_fate_table(mcmc_combined, "MCMC", grouped=True)
smc_counts = move_fate_table(smc_combined, "SMC", grouped=True)

In [ ]:
pd.DataFrame(mcmc_counts)

In [ ]:
# create latex tables with the outputs and have the highest percentage in each column bolded
# have the {move}_{fate} split up so that {move} is a heading with each {fate} as a subheading, and the values in the table are the counts with percentages in brackets
# embolden the highest percentage in each column and have the percentage appear below the count in the table, so that the table is more readable
def create_latex_table(counts, samplername):
    df = pd.DataFrame(counts)
    df = df.fillna("--")
    # split the columns into multiindex
    new_columns = []
    for col in df.columns:
        if col in ["proposal_mechanism", "ss_prop"]:
            new_columns.append((col, ""))
        else:
            move, fate = col.split("_")
            new_columns.append((move, fate))
    df.columns = pd.MultiIndex.from_tuples(new_columns)
    # bold the highest percentage in each column
    for move in MOVES:
        for fate in FATE_LABELS:
            col = (move, fate)
            if col in df.columns:
                percentages = df[col].str.extract(r"\(([\d\.]+)\)").astype(float)
                max_idx = percentages.idxmax()[0]
                df.loc[max_idx, col] = "\\textbf{" + df.loc[max_idx, col] + "}"
    latex_table = df.to_latex(index=False, escape=False, multicolumn=True, multicolumn_format='c', column_format='l' + 'c' * (len(df.columns) - 2), multirow=False)
    return latex_table

print(create_latex_table(mcmc_counts, "MCMC"))

## Acceptance by move type at the tree-size cap
How often each move type is accepted (MCMC) or kept (SMC) when the tree it was proposed on is at `max_tree_size` versus below it. At the cap `select_move` never proposes a grow, so there is no at-cap grow bar. A (Flat) HINTS sweep that starts at the cap can still draw a grow after an earlier prune in the same sweep frees a node; that grow was drawn on a tree below the cap and is counted there.

In [ ]:
# Acceptance by move type, split on whether the tree a move was proposed on was
# already at max_tree_size. At the cap a grow is barred outright, so the only
# way to restructure the tree is through prune and change -- this is how often
# those actually get through there, against the same moves below the cap.
#
# The tree a move was proposed on is the state *before* its sample() call:
#   MCMC: the previous iteration's state (the initial tree for iteration 0).
#   SMC:  the particle's ancestor after resampling, which the h5 does not
#         record. Systematic resampling hands out copies in ancestor order, one
#         assignment per interval of its uniform u, so the candidates are
#         enumerated from the stored weights and kept only if they agree with
#         what is known: a particle whose call stayed holds its ancestor's very
#         slot, and one that moved changed size by at most the grows/prunes its
#         call applied. A call whose surviving candidates disagree on whether
#         the ancestor was at the cap is left out, and counted.
# For a HINTS sweep "at the cap" means the sweep started there; a prune early in
# the sweep can free room for a grow later in it. That grow was drawn on a tree
# below the cap -- select_move gives grow probability 0 at the cap, judged on
# the sweep's working tree -- so every grow is counted as below the cap.
# MCMC rates are the outer accept; SMC never rejects, so its rate is "kept" --
# the move survived the proposal (see move_fates).

_GROW, _PRUNE = MOVE_CODE["grow"], MOVE_CODE["prune"]
_TOOK = (_pd.dg.PROPOSED, _pd.dg.APPLIED)


def _systematic_ancestors(logw):
    """Every ancestor vector systematic_resampling can draw from these
    weights: one per interval of u that changes the copy counts."""
    w = np.exp(logw - logw.max())
    w = (w / w.sum()).astype(np.float32)
    N = len(w)
    cdf = np.cumsum(w * N)
    prev = cdf - w * N
    breaks = np.unique(np.r_[0.0, np.mod(cdf, 1.0), 1.0])
    out = []
    for u in (breaks[:-1] + breaks[1:]) / 2:
        ncopies = (np.ceil(cdf - u) - np.ceil(prev - u)).astype(int)
        excess = ncopies.sum() - N
        if excess:   # check_stability's fix-up
            ncopies[np.nonzero(ncopies > 0)[0][-1]] -= excess
        if (ncopies >= 0).all():
            out.append(np.repeat(np.arange(N), ncopies))
    return out


def pre_call_at_cap(run, init_size, cap):
    """Per sample() call: 1 if the tree it was handed was at the cap, 0 if
    not, -1 if that can't be pinned down (SMC only, see above)."""
    slots = run["state_index"][()]
    sizes = run["state_tree_lengths"][()][slots]               # (T, N)
    T, N = slots.shape
    if "resampled" not in run:                                 # MCMC, N == 1
        pre = np.r_[init_size, sizes[:-1, 0]]
        return (pre >= cap).astype(np.int8)

    resampled = run["resampled"][()]
    logw = run["state_log_weights"][()].astype(np.float64)
    took = (run["call_outcome"][()] == _pd.dg.PROPOSED).reshape(T, N)
    call, move, outcome = run["call"][()], run["move"][()], run["outcome"][()]
    ok = np.isin(outcome, _TOOK)
    grows = np.bincount(call[ok & (move == _GROW)], minlength=T * N).reshape(T, N)
    prunes = np.bincount(call[ok & (move == _PRUNE)], minlength=T * N).reshape(T, N)
    lo = np.where(took, sizes - grows, sizes)
    hi = np.where(took, sizes + prunes, sizes)

    at_cap = np.full((T, N), -1, dtype=np.int8)
    at_cap[0] = init_size >= cap                               # all start on the init tree
    for t in range(1, T):
        prev_cap = sizes[t - 1] >= cap
        if not resampled[t]:
            at_cap[t] = prev_cap
            continue
        if prev_cap.all() or not prev_cap.any():               # every ancestor agrees
            at_cap[t] = prev_cap[0]
            continue
        stay = ~took[t]
        fits = [anc for anc in _systematic_ancestors(logw[t - 1])
                if (slots[t - 1, anc][stay] == slots[t][stay]).all()
                and ((sizes[t - 1, anc] >= lo[t]) & (sizes[t - 1, anc] <= hi[t])).all()]
        if not fits:
            continue
        flags = np.stack([prev_cap[anc] for anc in fits])
        agree = (flags == flags[0]).all(axis=0)
        at_cap[t, agree] = flags[0, agree]
    return at_cap.ravel()


def move_fate_counts_by_cap(experiment):
    """{True: at-cap table, False: below-cap table} in move_fate_counts' shape,
    pooled over every chain, plus how many moves were left out as unknown."""
    cap = experiment["max_tree_size"]
    init_size = len(experiment["init_tree"]) if experiment.get("init_tree") is not None else 0
    tables = {True: 0, False: 0}
    unknown = total = 0
    with h5py.File(experiment["h5_path"], "r") as f:
        runs = sorted((k for k in f if k.startswith("Run_")),
                      key=lambda k: int(k[len("Run_"):]))
        for name in runs:
            run = f[name]
            call = run["call"][()]
            per_move = pre_call_at_cap(run, init_size, cap)[call]
            per_move[run["move"][()] == _GROW] = 0     # never drawn on a full tree
            calls = {k: run[k][()] for k in ("call_id", "call_outcome", "accepted") if k in run}
            moves = {k: run[k][()] for k in ("move", "outcome", "dsurr")}
            moves["call"] = call
            for flag in (True, False):
                keep = per_move == flag
                sub = dict(calls, **{k: v[keep] for k, v in moves.items()})
                tables[flag] = tables[flag] + _pd.move_fates(sub)
            unknown += int((per_move < 0).sum())
            total += len(per_move)
    return tables, unknown, total


def _exp_label(experiment):
    p = experiment["proposal_mechanism"]
    return "MH" if p == "MH" else f"{p} {experiment['ss_prop']}"


def cap_acceptance_table(results):
    """One row per experiment: for each move type, the accept (MCMC) / kept
    (SMC) rate below the cap and at it, with how many moves that is out of.
    Also returns [(experiment, {True: at-cap, False: below-cap table})] for
    plot_cap_fates, so the h5 files are only read once."""
    rows, counts = [], []
    for experiment in results:
        tables, unknown, total = move_fate_counts_by_cap(experiment)
        counts.append((experiment, tables))
        row = {"experiment": _exp_label(experiment)}
        for flag, where in ((False, "below cap"), (True, "AT CAP")):
            grouped = group_fates(tables[flag])                # Accept, Reject, Barred, unavailable
            for m in ("grow", "prune", "change"):
                i = MOVE_CODE[m]
                n = grouped[i].sum()
                row[(m, where)] = f"{grouped[i, 0] / n:.2%} ({n:,})" if n else "--"
        row[("", "unknown")] = f"{unknown / total:.2%}" if total else "--"
        rows.append(row)
    df = pd.DataFrame(rows).set_index("experiment")
    df.columns = pd.MultiIndex.from_tuples(df.columns)
    return df, counts


CAP_MOVES = ("grow", "prune", "change")      # stay never reaches a fate worth splitting
CAP_BAR_W = 0.36                             # two bars per move, side by side
CAP_GROUPS = FATE_GROUPS[:3]                 # Accept, Reject, Barred -- no stay here


def _label_cap_bar(ax, x, shares, colours):
    """Each fate's share on its own segment, or, for a segment too thin to
    hold it, stacked above the bar in the segment's colour -- straight above
    rather than off to the side, so the two bars of a pair never cross.
    Accept (k == 0) is the exception: too thin to hold its share, it is
    anchored at the foot of the bar instead, so it always reads off the
    baseline it is measured from."""
    stroke = [pe.withStroke(linewidth=1.4, foreground="black")]
    y = 100.0 + GAP * 100.0 * 0.6
    for k, share in enumerate(shares):
        if share == 0:
            continue
        if k == 0 and share < INSIDE_MIN * 100.0:
            ax.text(x, 1.0, _pct(share), ha='center', va='bottom',
                    color=colours[k], size=10, weight='bold', zorder=4,
                    path_effects=stroke)
        elif share >= INSIDE_MIN * 100.0:
            ax.text(x, shares[:k].sum() + share / 2, _pct(share), ha='center',
                    va='center', color=colours[k], size=10, weight='bold',
                    zorder=4, path_effects=stroke)
        else:
            ax.text(x, y, _pct(share), ha='center', va='center', color=colours[k],
                    size=8.5, weight='bold', zorder=4, path_effects=stroke)
            y += GAP * 100.0


def plot_cap_fates(counts, samplername, ncols=3, footnote=None):
    """One panel per experiment, in plot_move_fates' style: for each move type
    a pair of 100% bars, below the cap and at it, stacked by Accept / Reject /
    Barred. A grow is never drawn at the cap, so that slot says so instead."""
    colours = {k: hexcode for k, (_, hexcode, _) in enumerate(CAP_GROUPS)}
    nrows = int(np.ceil(len(counts) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 6, nrows * 4.8),
                             squeeze=False)
    axes = axes.ravel()
    centres = np.arange(len(CAP_MOVES))
    offsets = {False: -CAP_BAR_W / 2 - 0.035, True: CAP_BAR_W / 2 + 0.035}

    # Headroom for the tallest stack of above-bar labels in any panel, shared
    # so every panel sits on the same 0-100 scale. Accept never goes above.
    n_above = 0
    for _, tables in counts:
        for flag in (False, True):
            g = group_fates(tables[flag])[:, :3]
            for m in CAP_MOVES:
                row = g[MOVE_CODE[m]]
                if row.sum():
                    shares = 100.0 * row / row.sum()
                    n_above = max(n_above, sum(1 for s in shares[1:] if 0 < s < INSIDE_MIN * 100))
    top = 100.0 * (1 + GAP * (n_above + 0.6))

    for ax, (experiment, tables) in zip(axes, counts):
        ticks, labels, n_props = [], [], 0
        for flag in (False, True):
            g = group_fates(tables[flag])[:, :3]
            for c, m in zip(centres, CAP_MOVES):
                x = c + offsets[flag]
                row = g[MOVE_CODE[m]]
                n = row.sum()
                n_props += n
                ticks.append(x)
                labels.append(f"{'BELOW CAP' if not flag else 'AT CAP'}\n{_si(n) if n else '–'}")
                if n == 0:
                    ax.bar(x, 100, width=CAP_BAR_W, color="none", edgecolor="#c7c2b8",
                           hatch="//", linewidth=0.8, zorder=1)
                    ax.text(x, 50, "never\nproposed", ha="center", va="center",
                            size=9, color="#8c8578", style="italic", zorder=4)
                    continue
                shares = 100.0 * row / n
                bottom = 0.0
                for k in range(3):
                    ax.bar(x, shares[k], width=CAP_BAR_W, bottom=bottom, zorder=2,
                           color=colours[k], edgecolor=SURFACE, linewidth=1.5)
                    bottom += shares[k]
                _label_cap_bar(ax, x, shares, colours)

        order = np.argsort(ticks)
        ax.set_xticks(np.array(ticks)[order], np.array(labels)[order], fontsize=9)
        # The move name once per pair, under its two bars' own labels.
        for c, m in zip(centres, CAP_MOVES):
            ax.annotate(m, xy=(c, 0), xycoords=("data", "axes fraction"),
                        xytext=(0, -38), textcoords="offset points",
                        ha="center", va="top", fontsize=11, fontweight="bold")

        ss = experiment['ss_prop']
        p = experiment['proposal_mechanism']
        setting = f"Hierarchy={ss}" if "HINTS" in p else ("ss_prop=None" if p == "MH" else f"ss_prop={ss}")
        ax.set_title(f"{p} ({setting}, {n_props:,} proposals)", fontsize=14, fontweight='bold')
        ax.set_xlim(-0.6, len(CAP_MOVES) - 0.4)
        ax.set_ylim(0, top)
        ax.set_yticks(np.arange(0, 101, 25))
        ax.set_ylabel('Share of proposals (%)')
        ax.grid(axis='x', visible=False)

    for ax in axes[len(counts):]:
        ax.set_visible(False)
    for ax in axes:
        if ax.get_subplotspec().colspan.start > 0:
            ax.set_ylabel('')
            ax.set_yticklabels([])

    fig.legend(handles=[Patch(facecolor=colours[k], label=name)
                        for k, (name, _, _) in enumerate(CAP_GROUPS)],
               loc='upper center', ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.985))
    experiment = counts[0][0]
    fig.suptitle(f"Move Outcomes Below vs At max_tree_size over {experiment['data']['n_runs']} chains "
                 f"in {samplername} experiments on {experiment['dataset']}",
                 fontsize=20, y=1, fontweight='bold')
    if footnote:
        fig.text(0.5, -0.01, footnote, ha='center', fontsize=10, fontstyle='italic')
    fig.tight_layout(rect=(0, 0.04, 1, 0.98), h_pad=2.5)
    plt.show()


CAP_NOTE = ("Each move is placed by the tree its sample() call started on; a grow is never "
            "drawn on a full tree, so a HINTS sweep's grow after an in-sweep prune counts as below cap.")

smc_results_reordered = [e for e in smc_results if e['proposal_mechanism'] == 'HINTS'] + \
                        [e for e in smc_results if e['proposal_mechanism'] == 'Flat HINTS'] + \
                        [e for e in smc_results if e['proposal_mechanism'] == 'DA'] + \
                        [e for e in smc_results if e['proposal_mechanism'] == 'MH']
mcmc_results_reordered = [e for e in mcmc_results if e['proposal_mechanism'] == 'HINTS'] + \
                        [e for e in mcmc_results if e['proposal_mechanism'] == 'Flat HINTS'] + \
                        [e for e in mcmc_results if e['proposal_mechanism'] == 'DA'] + \
                        [e for e in mcmc_results if e['proposal_mechanism'] == 'MH']

mcmc_cap_table, mcmc_cap_counts = cap_acceptance_table(mcmc_results_reordered)
smc_cap_table, smc_cap_counts = cap_acceptance_table(smc_results_reordered)
plot_cap_fates(mcmc_cap_counts, "MCMC", footnote=CAP_NOTE,ncols=2)
plot_cap_fates(smc_cap_counts, "SMC",
               footnote="SMC has no accept/reject step -- it reweights, so Reject here is only the "
                        "subsample screen (DA, HINTS) and Accept means the proposal survived.\n" + CAP_NOTE,ncols=2)
display(mcmc_cap_table)
display(smc_cap_table)

In [ ]:
# The counts above, turned around: one panel for moves proposed below
# max_tree_size and one for moves proposed at it, with every algorithm side by
# side within each move type -- so algorithms are compared directly, where the
# panels above compare below and at the cap within one algorithm.


def _wrapped_label(experiment):
    """A tick label one word per line, with the subsample setting -- ss_prop,
    or the HINTS hierarchy written without spaces so it fits under one bar --
    on its own line beneath the sampler's name."""
    p = experiment["proposal_mechanism"]
    lines = p.split()
    ss = experiment["ss_prop"]
    if p != "MH":
        lines.append(str(ss).replace(" ", ""))
    return "\n".join(lines)


def plot_cap_fates_by_move(counts, samplername, footnote=None):
    """`counts` is cap_acceptance_table's [(experiment, {True: at cap,
    False: below cap})]. Bars are 100% stacks of Accept / Reject / Barred,
    labelled as in plot_cap_fates: on the segment, or stacked above the bar
    for a segment too thin to hold its share."""
    colours = {k: hexcode for k, (_, hexcode, _) in enumerate(CAP_GROUPS)}
    n_exp = len(counts)
    pitch = 0.9 / n_exp                          # bar spacing within a move's group
    width = pitch * 0.85

    # Headroom for the tallest stack of above-bar labels, as in plot_cap_fates;
    # Accept is labelled at the foot of the bar, so it never needs any.
    n_above = max(sum(1 for s in (100.0 * row / row.sum())[1:] if 0 < s < INSIDE_MIN * 100)
                  for _, tables in counts for flag in (False, True)
                  for row in group_fates(tables[flag])[:, :3] if row.sum())
    top = 100.0 * (1 + GAP * (n_above + 0.6))

    fig, axes = plt.subplots(2, 1, figsize=(20, 14))
    for ax, (flag, where) in zip(axes, ((False, "Moves Proposed Below Max Tree Size Cap"),
                                        (True, "Moves Proposed at Max Tree Size Cap"))):
        ticks, labels = [], []
        for c, m in enumerate(CAP_MOVES):
            drawn = False
            for e, (experiment, tables) in enumerate(counts):
                x = c + (e - (n_exp - 1) / 2) * pitch
                ticks.append(x)
                labels.append(_wrapped_label(experiment))
                row = group_fates(tables[flag])[MOVE_CODE[m], :3]
                n = row.sum()
                if n == 0:
                    continue
                drawn = True
                shares = 100.0 * row / n
                bottom = 0.0
                for k in range(3):
                    ax.bar(x, shares[k], width=width, bottom=bottom, zorder=2,
                           color=colours[k], edgecolor=SURFACE, linewidth=1.2)
                    bottom += shares[k]
                _label_cap_bar(ax, x, shares, colours)
            if not drawn:
                ax.text(c, 50, f"{m.capitalize()} is never proposed\non a tree at the cap",
                        ha='center', va='center', size=13, style='italic', color="#8c8578")
            # The move's name over its group, and a rule between groups.
            ax.annotate(m.capitalize(), xy=(c, 1), xycoords=("data", "axes fraction"),
                        xytext=(0, 4), textcoords="offset points",
                        ha='center', va='bottom', size=15, weight='bold',style='italic')
            if c:
                ax.axvline(c - 0.5, color="#c7c2b8", linewidth=1.2, zorder=1)

        ax.set_xticks(ticks, labels, fontsize=10, weight='bold',style='italic')
        ax.set_xlim(-0.5, len(CAP_MOVES) - 0.5)
        ax.set_ylim(0, top)
        ax.set_yticks(np.arange(0, 101, 25))
        ax.set_ylabel("Share of proposals (%)", fontsize=12)
        # The row's heading, lifted clear of the move names above the groups.
        ax.set_title(where, fontsize=17, fontweight='bold', pad=30)
        ax.grid(axis='x', visible=False)

    fig.legend(handles=[Patch(facecolor=colours[k], label=name)
                        for k, (name, _, _) in enumerate(CAP_GROUPS)],
               loc='upper center', ncol=3, frameon=False, bbox_to_anchor=(0.5, 0.975))
    experiment = counts[0][0]
    fig.suptitle(f"Move Outcomes by Move Type, Below vs At max_tree_size, over "
                 f"{experiment['data']['n_runs']} chains in {samplername} experiments "
                 f"on {experiment['dataset']}", fontsize=20, y=1, fontweight='bold')
    if footnote:
        fig.text(0.5, -0.01, footnote, ha='center', fontsize=10, fontstyle='italic')
    fig.tight_layout(rect=(0, 0.02, 1, 0.965), h_pad=2.5)
    plt.show()


plot_cap_fates_by_move(mcmc_cap_counts, "MCMC", footnote=CAP_NOTE)
plot_cap_fates_by_move(smc_cap_counts, "SMC",
                       footnote="SMC has no accept/reject step -- it reweights, so Reject here is only the "
                                "subsample screen (DA, HINTS) and Accept means the proposal survived.\n" + CAP_NOTE)

## Proposal outcome traces
What each proposal came to, per iteration (MCMC) / step (SMC), stacked to 100%: accepted, rejected outright, passed stage 1 but rejected at stage 2, rejected at stage 1, or no valid move -- one call per chain or particle, a whole sweep for (Flat) HINTS. A second figure follows every move drawn inside the HINTS sweeps, in the same colours, plus moves applied by their block but undone below the top level.

In [ ]:
# What each proposal came to, traced over the run: at every iteration (MCMC) /
# step (SMC), the share of chains or particles in each state.
#
#   Accept                           the move -- or the HINTS sweep -- became
#                                    the new state
#   Reject                           an unscreened proposal turned down by the
#                                    outer MH step (MH, and DA's skip path)
#   Stage 1 accept + stage 2 reject  passed the subsample screen (DA) or every
#                                    level of the sweep (HINTS), then the outer
#                                    MH step on the full data turned it down
#   Stage 1 reject                   the screen threw it out: the state is
#                                    handed straight back, the full target never
#                                    evaluated
#   No valid move                    barred, inadmissible, or nothing to draw --
#                                    the state is handed back without a screen
#
# A (Flat) HINTS call is a sweep of many moves, one per block, so a second
# figure follows the moves inside the sweeps: the share of every move drawn
# that iteration, in the same colours read move by move --
#
#   Accept                           applied by its block, and the sweep landed
#   Stage 1 accept + stage 2 reject  applied, but the sweep was rejected at the
#                                    top
#   Applied, undone below the top    applied, but an intermediate level threw
#                                    out every move in the sweep, or the
#                                    finished sweep was barred
#   Stage 1 reject                   screened out by its own block
#   No valid move                    barred, inadmissible, or nothing to draw
#
# SMC has no stage 2 -- it reweights -- so neither Reject nor Stage 1 accept +
# stage 2 reject occurs there.
TRACE_STATES = (
    ("Accept",                          "#22A822"),
    ("Reject",                          "#e34948"),
    ("Stage 1 accept + stage 2 reject", "#ff7f0e"),
    ("Applied, undone below the top",   "#54c8e2"),
    ("Stage 1 reject",                  "#ff9896"),
    ("No valid move",                   "#8c8578"),
)
(T_ACCEPT, T_REJECT, T_S2_REJECT, T_UNDONE, T_S1_REJECT,
 T_NO_MOVE) = range(len(TRACE_STATES))

# What each state means for a HINTS move, for the legend.
HINTS_MEANING = {
    T_ACCEPT:    "HINTS: move landed",
    T_S2_REJECT: "HINTS: sweep rejected at the top",
    T_UNDONE:    "HINTS: block applied it, a level above undid it",
    T_S1_REJECT: "HINTS: move screened out by its block",
}


def call_states(run):
    """For one h5 run: the TRACE_STATES index of every sample() call, and of
    every move logged in it, with the call each move belongs to."""
    call = run["call"][()]
    outcome = run["outcome"][()]
    call_outcome = run["call_outcome"][()]
    n_calls = len(call_outcome)
    # Stays are "accepted" trivially by the outer step, so the flag is only
    # read for calls that proposed something. SMC never rejects.
    accepted = (run["accepted"][()].astype(bool) if "accepted" in run
                else np.ones(n_calls, dtype=bool))

    # dsurr is nan exactly when a move was not screened.
    screened = np.bincount(call, weights=~np.isnan(run["dsurr"][()]),
                           minlength=n_calls) > 0
    reached_screen = np.bincount(
        call, weights=(outcome == SCREENED_OUT) | (outcome == APPLIED),
        minlength=n_calls) > 0

    proposed = call_outcome == PROPOSED
    state = np.full(n_calls, T_NO_MOVE)
    state[proposed & accepted] = T_ACCEPT
    state[proposed & ~accepted & ~screened] = T_REJECT
    state[proposed & ~accepted & screened] = T_S2_REJECT
    state[call_outcome == SCREENED_OUT] = T_S1_REJECT               # DA
    # HINTS: a sweep with nothing left to propose is a STAY. If any of its
    # moves met a screen, the screens (or a level above them) emptied it.
    state[(call_outcome == STAY) & reached_screen] = T_S1_REJECT

    # A block-applied move's fate is its sweep's: it lands only if the sweep
    # does, and is lost at the top if the sweep is rejected there.
    move_state = np.full(len(call), T_NO_MOVE)
    move_state[outcome == SCREENED_OUT] = T_S1_REJECT
    applied = outcome == APPLIED
    sweep = state[call]
    move_state[applied] = T_UNDONE
    move_state[applied & (sweep == T_ACCEPT)] = T_ACCEPT
    move_state[applied & (sweep == T_S2_REJECT)] = T_S2_REJECT
    return state, call, move_state


def outcome_traces(experiment):
    """Per iteration (MCMC) / step (SMC), summed over chains, as
    (len(TRACE_STATES), n) counts: of calls in each state, and of moves in
    each state. Also the number of chains."""
    with h5py.File(experiment["h5_path"], "r") as f:
        runs = sorted((k for k in f if k.startswith("Run_")),
                      key=lambda k: int(k[len("Run_"):]))
        per_run = []
        for name in runs:
            run = f[name]
            state, call, move_state = call_states(run)
            if "calls_at_step" in run:
                # SMC: one call per particle per step, in step order.
                step = np.searchsorted(run["calls_at_step"][()],
                                       np.arange(len(state)), side="right")
            else:
                step = np.arange(len(state))
            per_run.append((step, state, step[call], move_state))

    n = max(step.max() for step, *_ in per_run) + 1
    size = len(TRACE_STATES) * n
    states = np.zeros((len(TRACE_STATES), n))
    moves = np.zeros((len(TRACE_STATES), n))
    for step, state, move_step, move_state in per_run:
        states += np.bincount(state * n + step, minlength=size).reshape(-1, n)
        moves += np.bincount(move_state * n + move_step, minlength=size).reshape(-1, n)
    return states, moves, len(runs)


def _smooth(y, window):
    """Centred moving average along the last axis, averaging over only the
    points that exist at the edges."""
    if window <= 1:
        return y
    kernel = np.ones(window)
    num = np.apply_along_axis(np.convolve, -1, y, kernel, "same")
    return num / np.convolve(np.ones(y.shape[-1]), kernel, "same")


def _trace_title(experiment):
    p, ss = experiment["proposal_mechanism"], experiment["ss_prop"]
    if p == "MH":
        return "Metropolis-Hastings"
    if "HINTS" in p:
        return f"{p} (Hierarchy={ss})"
    return f"{p} (ss_prop={ss})"


def _stack_figure(panels, title, legend_meaning, footnote, ncols):
    """One figure of 100% stacks. `panels` is [(experiment, counts, subtitle,
    ylabel, xlabel)], counts a (len(TRACE_STATES), n) array per iteration.
    Only the states that occur go in the legend, each followed by its entry
    in `legend_meaning` where it has one."""
    nrows = int(np.ceil(len(panels) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 5.4, nrows * 3.8),
                             squeeze=False,sharey=True)
    axes = axes.ravel()
    for ax, (experiment, counts, subtitle, ylabel, xlabel) in zip(axes, panels):
        n = counts.shape[1]
        share = 100.0 * counts / np.maximum(counts.sum(axis=0), 1e-12)
        ax.stackplot(np.arange(n), share, colors=[c for _, c in TRACE_STATES],
                     linewidth=0,alpha=0.75)
        ax.set_xlim(0, n - 1)
        ax.set_ylim(0, 100)
        ax.set_title(f"{_trace_title(experiment)}\n{subtitle}",
                     fontsize=12, fontweight="bold")
        if ax.get_subplotspec().rowspan.start == nrows - 1:
            ax.set_xlabel(xlabel)
        if ax.get_subplotspec().colspan.start == 0:
            ax.set_ylabel(ylabel)
        # ax.grid(False)
    for ax in axes[len(panels):]:
        ax.set_visible(False)

    handles = []
    for k, (name, colour) in enumerate(TRACE_STATES):
        if not any(p[1][k].any() for p in panels):
            continue
        if k in legend_meaning:
            name = f"{name} ({legend_meaning[k]})"
        handles.append(Patch(facecolor=colour, label=name))
    n_rows = 1 if len(handles) <= 4 else 2
    fig.legend(handles=handles, loc="upper center", frameon=False,
               ncol=int(np.ceil(len(handles) / n_rows)),
               bbox_to_anchor=(0.5, 0.95))
    fig.suptitle(title, fontsize=18, fontweight="bold", y=0.97)
    if footnote:
        fig.text(0.5, -0.01, footnote, ha="center", va="top", fontsize=10,
                 fontstyle="italic")
    fig.tight_layout(rect=(0, 0.02, 1, 0.97 - 0.02 * (n_rows - 1)))
    plt.show()


def plot_outcome_traces(results, samplername, smooth=0.005, ncols=3, footnote=None):
    """Two figures of 100% stacks per iteration: the share of calls in each
    TRACE_STATES state for every experiment, then for (Flat) HINTS the share
    of every move drawn in the sweeps. `smooth` is the moving-average window as
    a fraction of each run's length (0 for none)."""
    traces = [outcome_traces(e) for e in results]
    step_name = "Step" if samplername == "SMC" else "Iteration"
    unit = "particles" if samplername == "SMC" else "chains"

    calls, moves = [], []
    for experiment, (states, move_counts, n_runs) in zip(results, traces):
        n = states.shape[1]
        window = max(1, int(round(smooth * n)))
        subtitle = f"{n_runs} runs, window {window}"
        calls.append((experiment, _smooth(states, window), subtitle,
                      f"Share of {unit} (%)", step_name))
        if "HINTS" in experiment["proposal_mechanism"]:
            per_sweep = move_counts.sum() / states.sum()
            moves.append((experiment, _smooth(move_counts, window),
                          f"{subtitle}, {per_sweep:.1f} moves per sweep",
                          "Share of moves drawn (%)", step_name))

    _stack_figure(calls, f"Outcome of each proposal per {step_name.lower()} -- "
                         f"{samplername} experiments on {results[0]['dataset']}",
                  {}, footnote, ncols)
    if not moves:
        return
    notes = [footnote] if footnote else []
    notes.append(
        "Every move drawn in the sweeps, not every sweep. A move applied by its block lands "
        "only if its sweep is accepted at the top. Hierarchical HINTS does not log\nwhich "
        "moves an intermediate level undid in a sweep that still went on, so its Accept share "
        "is an upper bound and its \"undone below the top\" share a lower one; for Flat "
        "HINTS both are exact.")
    _stack_figure(moves, f"Outcome of each move in a HINTS sweep per {step_name.lower()} -- "
                         f"{samplername} experiments on {results[0]['dataset']}",
                  HINTS_MEANING, "\n".join(notes), ncols)


plot_outcome_traces(mcmc_combined, "MCMC",ncols=2)
plot_outcome_traces(smc_combined, "SMC",ncols=2,
                    footnote="SMC has no stage 2 -- it reweights -- so every proposal "
                             "that survives the proposal is kept, and Reject never occurs.")